In [ ]:
from src.Functions_General import *
from src.Functions_Energy_Model import *
from src.Functions_Grid_Simulator_1 import *
from src.Functions_Grid_Simulator_2 import *
from src.Functions_Load_Emulator_and_DSM import *

---

### **1. Create the network**

This function is used to create the network to be simulated. 

All data needed to create the network must be provided via the Excel file contained in the path: 

📂 *files/grid/0. input_data*

The Excel file must contain the following sheets:

- **load**: This sheet contains the same information as the `user CACER.xlsx` file. For each network user, all the information needed to model their energy flows must be provided. During the initialization phase of the network simulator, the defined users will be simulated, and for each, the net exchanges with the grid will be calculated—that is, the difference between injections and withdrawals at any given moment in time. The users' exchanges will then serve as input for the power flow analysis. In addition to the information contained in the user CACER.xlsx file, the following data must be provided for each user:
    
    - *name*: user identifier in the network simulator

    - *bus_name*: identifier of the bus to which the user is connected (the same identifiers provided in the `bus sheet` must be used)

    - *power_factor*: specify the typical power factor of the user being simulated

    - *id_pod*: ....

- **bus**: For each bus in the network the following information must be provided

    - *name*: bus identifier in the network simulator

    - *base kv*: bus voltage in kilovolts

    - *vmax*: maximum allowable voltage in p.u.max

    - *vmin*: minimum allowable voltage in p.u.

    - *type*: Type of the bus. “n” - node, “b” - busbar, “m” - muff

    - *x*: latitude, useful for network plotting

    - *y*: longitude, useful for network plotting

    - *id_bus*: numerical bus identifier in the network simulator

    - *user_type*: user type (i.e., consumer, prosumer, no_load, etc.), usefule for network plotting

- **ext_grid**: For the external node (`slack node`) the following information must be provided

    - *name* slack node identifier in the network simulator

    - *bus* identifier of the bus to which the slack node is connected (the same identifier provided in the `bus sheet` must be used)

    - *voltage* voltage of the slack node in p.u. (always equal to 1)

    - *s_sc_max_mva* Maximum short circuit apparent power to calculate internal impedance of ext_grid for short circuit calculations

    - *s_sc_min_mva* Minimum short circuit apparent power to calculate internal impedance of ext_grid for short circuit calculations

    - *rx_max* Maximal R/X-ratio to calculate internal impedance of ext_grid for short circuit calculations (actually deactivated in the `create_ext_grid()` function)

    - *rx_min* Minimum R/X-ratio to calculate internal impedance of ext_grid for short circuit calculations (actually deactivated in the `create_ext_grid()` function)	


- **branch(line)**: For each line in the network the following information must be provided

    - *name* line identifier in the network simulator

    - *name from* id of the bus on one side which the line will be connected with

    - *name to* id of the bus on the other side which the line will be connected with

    - *r [ohm]* line resistance in ohm

    - *x [ohm]* line reactance in ohm

    - *rate [kA]* line capacitance in nano Farad

**Attention:** 

*Other information about the lines settings are provided in the function `create_lines()`*

- **branch(trafo)**: For each trafo in the network the following information must be provided

    - *name*: transformer identifier in the network simulator
    
    - *name from*: the bus on the high - voltage side on which the trasformer will be connected (FromBus)

    - *name to*: the bus on the low - voltage side on which the trasformer will be connected (ToBus)

    - *short circuit [%]*: relative short-circuit voltage (Irated_trafo_side / Isc_max_trafo_side)

    - *short circuit power factor*: real part of relative short-circuit voltage (sc_voltage_perc * sc_pow_factor)

    - *short circuit resistance percentage [%]*: actually deactivated

    - *Snom [MVA]*: rated apparent power (S = radq(P^2 + Q^2))

    - *vn_hv_kv*: voltage of the high voltage bus (actually deactivated)

    - *vn_lv_kv*: voltage of the low voltage bus (actually deactivated)

    - *pfe_kw*: iron losses [kW]

    - *pcu_kw*: copper losses in kW (actually deactivated)

    - *i0_percent*: open loop losses in percent of rated current (I0)

**Attention:** 

*Other information about the trafos settings are provided in the function `create_trafos()`*

- **switch**: For each switch in the network the following information must be provided

    - *name*: switch identifier in the network simulator

    - *bus name*: bus where is connected the switch

    - *element*: index of the element, bus id if et == “b” (bus - bus switch), line id if et == “l” (bus-line switch), trafo id if et == “t” (bus-trafo switch)

    - *et*: element type, “l” = switch between bus and line, “t” = switch between bus and transformer, “b” = switch between two buses

    - *stato*: switch position: False = open, True = closed

**Attention:** 

*More information about the parameters of the various network components is available at the following link https://pandapower.readthedocs.io/en/v2.2.1/elements.html*

In addition to creating the network object, a network file (in format `name.p`) is saved in the folder: 

📂 *files/grid/1. network*

In [ ]:
# we can create the network from an external file setted in config file
network = create_network(name = "example", f_hz=50)

---

#### 1.1 Plot network

The topology of the created network is plotted. If no bus coordinates have been provided, artificial coordinates will be created automatically.

In [ ]:
title = "Example_network" # the plot will be saved in an html format using this title (folder: assets\grid\)
plot_network(network, title)

The network topology is plotted, highlighting the various voltage levels around buses and lines.

In [ ]:
title = "Example_network - vlevel" # the plot will be saved in an html format using this title (folder: assets\grid\)
plot_network_vlevel(network, title)

**Attention:**

Other useful plot functions are contained in the file *Functions_Grid_Simulator_v1.py*

---

### **2. Initialize grid simulation and export users energy exchange with the grid**

Based on the information provided, all the energy flows of the various users are created and the net exchanges with the grid are calculated as the difference between energy taken and energy injected.

Exchange profiles are saved in a csv file at the following path:

📂 *files/energy/user_type_energy_exchange.csv*

In [ ]:
initialize_load_flow_simulator()

---

### **3. Power Flow simulation**

A power flow simulation is performed for each instant of time for an entire year. Simulations are performed on an hourly basis but this parameter can be changed in the load_time_series() function.

The power flow analysis results are saved in a pickle file at the following path:

📂 *files/grid/2. results/case_denomination.pkl*

At the moment the following simulation results are collected and saved:

- **extgridActivePowerTime**:  
  Active power exchanged by the external grid (*ext_grid*) at each simulation timestep.  
  Positive values typically indicate power supplied by the external grid to the network.  
  **Unit:** MW

- **extgridReactivePowerTime**:  
  Reactive power exchanged by the external grid (*ext_grid*) at each simulation timestep.  
  Positive values generally indicate reactive power injection into the network.  
  **Unit:** MVAr

- **busVoltageTime**:  
  Voltage magnitude calculated at each bus for every simulation timestep.  
  In pandapower, bus voltages are usually expressed in per-unit relative to the nominal bus voltage.  
  **Unit:** p.u.

- **busActivePowerTime**:  
  Net active power balance at each bus over time.  
  Represents the algebraic sum of active power injections and consumptions connected to the bus.  
  **Unit:** MW

- **busReactivePowerTime**:  
  Net reactive power balance at each bus over time.  
  Represents the algebraic sum of reactive power injections and consumptions connected to the bus.  
  **Unit:** MVAr

- **lineLoadingTime**:  
  Loading percentage of each line at every timestep.  
  Computed with respect to the maximum allowable current or apparent power rating of the line.  
  **Unit:** %

- **lineActiveLossesTime**:  
  Active power losses occurring on each line over time due to line impedance.  
  Calculated as the difference between active power at the sending and receiving ends of the line.  
  **Unit:** MW

- **lineReactiveLossesTime**:  
  Reactive power losses occurring on each line over time due to line reactance.  
  **Unit:** MVAr

- **lineActivePowerTimeTo**:  
  Active power flowing into the "to" side of each line at every timestep.  
  Corresponds to the pandapower result variable typically associated with `p_to_mw`.  
  **Unit:** MW

- **lineReactivePowerTimeTo**:  
  Reactive power flowing into the "to" side of each line at every timestep.  
  Corresponds to the pandapower result variable typically associated with `q_to_mvar`.  
  **Unit:** MVAr

- **lineActivePowerTimeFrom**:  
  Active power flowing from the "from" side of each line at every timestep.  
  Corresponds to the pandapower result variable typically associated with `p_from_mw`.  
  **Unit:** MW

- **lineReactivePowerTimeFrom**:  
  Reactive power flowing from the "from" side of each line at every timestep.  
  Corresponds to the pandapower result variable typically associated with `q_from_mvar`.  
  **Unit:** MVAr

- **trafoLoadingTime**:  
  Transformer loading percentage at each timestep.  
  Represents the transformer utilization with respect to its rated apparent power.  
  **Unit:** %

- **trafoActiveLossesTime**:  
  Active power losses in transformers over time, including copper and core losses.  
  **Unit:** MW

- **trafoReactiveLossesTime**:  
  Reactive power losses associated with transformer operation over time.  
  **Unit:** MVAr

- **loadPowerTime**:  
  Active power consumption of loads at each timestep.  
  Represents the demand profile applied to the network during the time-series simulation.  
  **Unit:** MW 

**Attention:**

*More information are available at the following link:*

https://pandapower.readthedocs.io/en/latest/powerflow.html?utm_source=#

Search the documentation for a specific network component and there you will find also the description of each single output result.

In [ ]:
case_denomination = "example_case" # the result will be saved in a pkl format using this title (folder: files\grid\2. results\)

results_dict = load_flow_simulator(case_denomination, network)

---
---